# Severity Prediction

## Purpose
Predict whether a COVID-19 VAERS adverse event report will be classified as **serious** (`SERIOUS = 1`) using structured clinical features and free-text symptom narratives.

## Models compared
- Logistic Regression (with TF-IDF text + structured features)
- Decision Tree
- Random Forest

## Data flow
1. Load pre-imputed VAERS dataset 
2. Load precomputed comorbidity indicators
3. Clean symptom text — separate pipelines for supervised (leakage-scrubbed) and clustering tasks
4. Build feature matrix: numeric + categorical + manufacturer + comorbidity + TF-IDF text
5. Train models with stratified k-fold cross-validation and grid search
6. Compare structured-only vs structured+text feature sets
7. Extract and save feature importance for Logistic Regression, Random Forest, and Decision Tree


## 1. Setup & Imports

In [1]:
# 1. SETUP

import os
import re
import sys
import json
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import scipy.sparse as sp

warnings.filterwarnings("ignore")

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

Python executable: /Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/bin/python
Python version: 3.12.10
Pandas version: 2.2.3


## 2. Load Data

Load the preprocessed VAERS dataset produced by `Data preparation.ipynb` (expected at `Outputs/df_clean_imputed.csv` or `Outputs/df_clean_engineered.csv`). If not found, re-run that notebook first.

In [2]:
# 2. LOAD PREPROCESSED DATASET
from pathlib import Path

HERE = Path().resolve()  # project root (the folder containing this notebook)
candidate_paths = [
    str(HERE / "Outputs" / "df_clean_imputed.csv"),
    str(HERE / "Outputs" / "df_clean_engineered.csv"),
]

data_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if data_path is not None:
    df_clean = pd.read_csv(data_path, low_memory=False)
    print(f"✓ Loaded dataset from: {data_path}")
    print(f"  Shape: {df_clean.shape}")
else:
    raise FileNotFoundError(
        "Expected one of these files but none were found:\n"
        + "\n".join(candidate_paths)
    )

✓ Loaded dataset from: /Users/ariellerothman/Desktop/Capstone/Outputs/df_clean_imputed.csv
  Shape: (986096, 113)


## 3. Load Precomputed Comorbidity Indicators

In [3]:
# ============================================================
# 3. LOAD PRECOMPUTED COMORBIDITY INDICATORS
# ============================================================

import os
import pandas as pd

from pathlib import Path
HERE   = Path().resolve()  # project root
OUTDIR = str(HERE / "Outputs")
os.makedirs(OUTDIR, exist_ok=True)

COMORB_PATH = str(HERE / "Outputs" / "comorbidity_indicators.csv")

# Detect currently available indicator columns in df_clean
existing_indicator_cols = [
    c for c in df_clean.columns
    if "__" in c or c.endswith("_MISSING")
]

print(f"Existing indicator columns already in df_clean: {len(existing_indicator_cols)}")

if not os.path.exists(COMORB_PATH):
    raise FileNotFoundError(
        f"Comorbidity indicator file not found: {COMORB_PATH}\n"
        "Run Data preparation notebook first to generate comorbidity_indicators.csv."
    )

comorb_df = pd.read_csv(COMORB_PATH, low_memory=False)

if "VAERS_ID" not in comorb_df.columns:
    raise ValueError("comorbidity_indicators.csv must contain VAERS_ID for merging.")

indicator_cols = [
    c for c in comorb_df.columns
    if c != "VAERS_ID" and ("__" in c or c.endswith("_MISSING"))
]

missing_in_df = [c for c in indicator_cols if c not in df_clean.columns]

if not missing_in_df:
    print("All comorbidity indicator columns are already present. Skipping merge.")
else:
    merge_cols = ["VAERS_ID"] + missing_in_df

    # Use one row per VAERS_ID to avoid accidental row multiplication
    comorb_merge = comorb_df[merge_cols].drop_duplicates(subset=["VAERS_ID"])

    df_clean = df_clean.merge(
        comorb_merge,
        on="VAERS_ID",
        how="left",
        validate="m:1"
    )

    # Fill unmatched IDs as 0 and cast to compact integer dtype
    for c in missing_in_df:
        df_clean[c] = df_clean[c].fillna(0).astype("int8")

    print(f"Merged {len(missing_in_df)} indicator columns from comorbidity_indicators.csv")

final_indicator_cols = [
    c for c in df_clean.columns
    if "__" in c or c.endswith("_MISSING")
]

print(f"Total indicator columns now in df_clean: {len(final_indicator_cols)}")

print("Sample indicator columns:")
print(final_indicator_cols[:20])

Existing indicator columns already in df_clean: 64
All comorbidity indicator columns are already present. Skipping merge.
Total indicator columns now in df_clean: 64
Sample indicator columns:
['MANU__JANSSEN', 'MANU__MODERNA', 'MANU__NOVAVAX', 'MANU__PFIZER_BIONTECH', 'MANU__UNKNOWN_MANUFACTURER', 'AGE_YRS_MISSING', 'HOSPDAYS_MISSING', 'NUMDAYS_MISSING', 'ONSET_INTERVAL_MISSING', 'MAX_DOSE_MISSING', 'HISTORY_MISSING', 'HISTORY__cardiovascular', 'HISTORY__respiratory', 'HISTORY__metabolic', 'HISTORY__endocrine_thyroid', 'HISTORY__musculoskeletal', 'HISTORY__mental_health', 'HISTORY__gastrointestinal', 'HISTORY__kidney', 'HISTORY__clotting']


## 4. Symptom Text Preprocessing


| Column | Purpose |
|---|---|
| `SYMPTOM_TEXT_CLEAN` | Base cleaning (lowercased, URLs/numbers removed) |
| `SYMPTOM_TEXT_CLEAN_NOLEAK` | Supervised modelling — leakage terms scrubbed (hospitalization, death, outcome labels) |

In [4]:
# ============================================================
# 4. SYMPTOM TEXT PREPROCESSING
# ============================================================
# Produces two text columns for downstream use:
# - SYMPTOM_TEXT_CLEAN: base cleaning only
# - SYMPTOM_TEXT_CLEAN_NOLEAK: leakage-scrubbed for supervised modelling

import re
import pandas as pd

# --------------------------------------------------
# Base text cleaning
# --------------------------------------------------
_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x) -> str:
    """
    Base-level cleaning for symptom text before any model-specific scrubbing.

    Steps:
        1. Lowercase and strip whitespace
        2. Remove URLs and email addresses
        3. Replace numeric tokens with placeholder '__NUM__'
        4. Remove non-alphanumeric characters (except hyphens)
        5. Collapse multiple whitespace

    Args:
        x: Raw symptom text string (or NaN).

    Returns:
        Cleaned lowercase string, or empty string if input is null.
    """
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# --------------------------------------------------
# Strong supervised leakage scrubbing for severity prediction
# Removes direct outcome / adjudication / downstream-care terms
# plus residual administrative/report-style phrases.
#
# WHY: The VAERS SERIOUS label is defined by hospitalization, death,
# disability, and life-threatening events. If those words appear
# verbatim in the symptom text, any classifier trivially learns to
# predict the label from them rather than real clinical patterns.
# --------------------------------------------------
LEAKAGE_PATTERNS = [re.compile(p) for p in [

    # ----------------------------------------------
    # Hospital / admission / downstream care
    # ----------------------------------------------
    r"\bhospital\w*\b",
    r"\badmit\w*\b",
    r"\binpatient\b",
    r"\bdischarg\w*\b",
    r"\bed\b",
    r"\ber\b",
    r"\bemergency\b",
    r"\bemergency room\b",
    r"\burgent\b",
    r"\burgent care\b",
    r"\btransport\w*\b",
    r"\btransfer\w*\b",
    r"\bambulance\b",
    r"\bems\b",
    r"\bhospice\b",

    # ----------------------------------------------
    # Death / life-threatening / rescue care
    # ----------------------------------------------
    r"\bdeath\b",
    r"\bdied\b",
    r"\bdead\b",
    r"\bfatal\b",
    r"\bpassed away\b",
    r"\bautopsy\b",
    r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b",
    r"\bventilat\w*\b",
    r"\bicu\b",
    r"\barrest\b",

    # ----------------------------------------------
    # Seriousness / adjudication labels
    # ----------------------------------------------
    r"\bserious\w*\b",
    r"\bnon[-\s]?serious\w*\b",
    r"\bmedically significant\b",
    r"\bmedically important\b",
    r"\bcriterion\b",
    r"\bcriteria\b",
    r"\boutcome\b",
    r"\bdisability\b",
    r"\bpermanent\b",
    r"\bprolonged\b",

    # ----------------------------------------------
    # Administrative / report-template phrasing
    # ----------------------------------------------
    r"\breported cause\b",
    r"\breport medically\b",
    r"\bcriterion hospitalization\b",
    r"\bhospitalization medically\b",
    r"\bsignificant outcome\b",
    r"\bnarrative\b",
    r"\breport\b",
    r"\breported\b",
    r"\bspontaneous\b",
    r"\bcase\b",
    r"\bsource\b",
    r"\bpatient unknown\b",
    r"\bunknown\b",

    # ----------------------------------------------
    # Residual report-style / quasi-administrative phrases
    # seen in current model outputs
    # ----------------------------------------------
    r"\bevent resulted\b",
    r"\bevents resulted\b",
    r"\bresulted\b",
    r"\bcaused prolonged\b",
    r"\bmedical center\b",
    r"\bclinic care\b",
    r"\bwent care\b",
    r"\bvisited\b",
    r"\bperformed\b",

    # ----------------------------------------------
    # Other report-level phrasing seen previously
    # ----------------------------------------------
    r"\broom\b",
    r"\bdepartment\b",
    r"\bvisit\b",
    r"\bwent room\b",
    r"\bwent doctor\b",
    r"\bdoctor visit\b"
]]

def scrub_leakage_terms_strong(x) -> str:
    """
    Remove supervised leakage terms from cleaned symptom text.

    Args:
        x: Cleaned symptom text string (output of clean_symptom_text).

    Returns:
        Text with leakage terms replaced by spaces and whitespace collapsed.
    """
    if pd.isna(x) or x == "":
        return ""
    x = str(x)
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(scrub_leakage_terms_strong)
)

print("Text columns created:")
print([c for c in df_clean.columns if "SYMPTOM" in c])


Text columns created:
['SYMPTOM_TEXT', 'SYMPTOM_TEXT_CHAR_LEN', 'SYMPTOM_TEXT_CLEAN', 'SYMPTOM_TEXT_CLEAN_NOLEAK']


## 5. Supervised Feature Matrix

Define which columns from `df_clean` enter the model as features. 


In [5]:
# 5. SUPERVISED FEATURE MATRIX SETUP
#     Cleaned final version for severity prediction

TEXT_FEATURE = "SYMPTOM_TEXT_CLEAN_NOLEAK"
y = df_clean["SERIOUS"].astype(int)

# Numeric features
# Keep onset-timing variables for the main model
# Remove HOSPDAYS because it is downstream / leakage-prone
numeric_cols = [
    c for c in ["AGE_YRS", "NUMDAYS", "ONSET_INTERVAL", "MAX_DOSE"]
    if c in df_clean.columns
]

# Coerce selected numeric columns to numeric dtype.
# This handles string artifacts like "UNKNOWN" from upstream exports.
for c in numeric_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

# Categorical features
categorical_cols = [
    c for c in ["SEX", "STATE", "YEAR", "MONTH"]
    if c in df_clean.columns
]

# Ensure categoricals are treated consistently as strings for the categorical imputer/encoder
for c in categorical_cols:
    df_clean[c] = df_clean[c].astype("string")

# Manufacturer and dose-related features
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]

dose_cols = [
    c for c in ["DOSE_COUNT", "MULTI_DOSE", "UNKNOWN_DOSE", "MULTI_MANUFACTURER"]
    if c in df_clean.columns
]

# Missingness indicators
missing_cols = [
    c for c in [
        "NUMDAYS_MISSING",
        "ONSET_INTERVAL_MISSING",
        "AGE_YRS_MISSING",
        "HISTORY_MISSING",
        "CUR_ILL_MISSING",
        "ALLERGIES_MISSING",
        "PRIOR_VAX_MISSING",
        "LAB_DATA_MISSING",
        "OTHER_MEDS_MISSING"
    ]
    if c in df_clean.columns
]

# Engineered clinical / comorbidity binary features
comorb_prefixes = (
    "HISTORY__",
    "CUR_ILL__",
    "ALLERGIES__",
    "PRIOR_VAX__",
    "LAB_DATA__",
    "OTHER_MEDS__"
)

comorb_cols = [c for c in df_clean.columns if c.startswith(comorb_prefixes)]

# Explicit exclusions for leakage-prone / helper / raw columns
exclude_cols = {
    # target / direct leakage
    "SERIOUS",
    "DIED",
    "L_THREAT",
    "ER_VISIT",
    "ER_ED_VISIT",
    "HOSPITAL",
    "DISABLE",
    "BIRTH_DEFECT",
    "HOSPDAYS",
    "DATEDIED",

    # identifiers / raw dates
    "VAERS_ID",
    "RECVDATE",
    "VAX_DATE",
    "ONSET_DATE",

    # raw free-text fields (already engineered elsewhere)
    "SYMPTOM_TEXT",
    "LAB_DATA",
    "OTHER_MEDS",
    "CUR_ILL",
    "HISTORY",
    "PRIOR_VAX",
    "ALLERGIES",

    # normalized helper text columns
    "HISTORY_NORM",
    "CUR_ILL_NORM",
    "ALLERGIES_NORM",
    "PRIOR_VAX_NORM",
    "LAB_DATA_NORM",
    "OTHER_MEDS_NORM",

    # text/clustering helper columns not used as structured inputs
    "SYMPTOM_TEXT_CLEAN",
    "SYMPTOM_TEXT_CLEAN_CLUSTER",
    "CLUSTER_TOKEN_COUNT",

    # optional: usually unhelpful if dataset is already one vaccine type
    "VAX_TYPE"
}

# Final structured feature list
structured_cols = (
    numeric_cols +
    categorical_cols +
    manufacturer_cols +
    dose_cols +
    missing_cols +
    comorb_cols
)

# Remove anything explicitly excluded
structured_cols = [c for c in structured_cols if c not in exclude_cols]

# Remove duplicates while preserving order
structured_cols = list(dict.fromkeys(structured_cols))

# Diagnostics
print("TEXT_FEATURE:", TEXT_FEATURE)
print("Target:", "SERIOUS")
print()

print("Numeric cols:", numeric_cols)
print("Categorical cols:", categorical_cols)
print("Manufacturer cols:", manufacturer_cols)
print("Dose cols:", dose_cols)
print("Missing cols:", missing_cols)
print("Comorbidity/clinical cols count:", len(comorb_cols))
print("Total structured feature count:", len(structured_cols))
print()

if numeric_cols:
    print("Numeric dtypes after coercion:")
    print(df_clean[numeric_cols].dtypes)

TEXT_FEATURE: SYMPTOM_TEXT_CLEAN_NOLEAK
Target: SERIOUS

Numeric cols: ['AGE_YRS', 'NUMDAYS', 'ONSET_INTERVAL', 'MAX_DOSE']
Categorical cols: ['SEX', 'STATE', 'YEAR', 'MONTH']
Manufacturer cols: ['MANU__JANSSEN', 'MANU__MODERNA', 'MANU__NOVAVAX', 'MANU__PFIZER_BIONTECH', 'MANU__UNKNOWN_MANUFACTURER']
Dose cols: ['DOSE_COUNT', 'MULTI_DOSE', 'UNKNOWN_DOSE', 'MULTI_MANUFACTURER']
Missing cols: ['NUMDAYS_MISSING', 'ONSET_INTERVAL_MISSING', 'AGE_YRS_MISSING', 'HISTORY_MISSING', 'CUR_ILL_MISSING', 'ALLERGIES_MISSING', 'PRIOR_VAX_MISSING', 'LAB_DATA_MISSING', 'OTHER_MEDS_MISSING']
Comorbidity/clinical cols count: 48
Total structured feature count: 74

Numeric dtypes after coercion:
AGE_YRS           float64
NUMDAYS           float64
ONSET_INTERVAL    float64
MAX_DOSE          float64
dtype: object


## 6. Column Inspection (Diagnostic)

Displays all columns in `df_clean` with their dtypes. Use this to verify that comorbidity indicators, text variants, and other engineered columns were created correctly before building the feature matrix.

In [6]:
# 6. COLUMN INSPECTION (DIAGNOSTIC)
import pandas as pd

print("Total columns:", len(df_clean.columns))

pd.set_option("display.max_rows", None)

col_df = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes
})

display(col_df)

Total columns: 115


,column,dtype
VAERS_ID,VAERS_ID,int64
DIED,DIED,int64
L_THREAT,L_THREAT,int64
ER_VISIT,ER_VISIT,int64
ER_ED_VISIT,ER_ED_VISIT,int64
HOSPITAL,HOSPITAL,int64
DISABLE,DISABLE,int64
BIRTH_DEFECT,BIRTH_DEFECT,int64
RECVDATE,RECVDATE,object
VAX_DATE,VAX_DATE,object


## 7. Model Training: CV-Based Comparative Analysis + Final Held-Out Test

A stratified working sample of 200,000 rows is drawn from the full dataset to keep runtimes tractable while preserving the class distribution (~19.85% serious).

For each model, we use a two-stage design to separate model selection from model comparison and final confirmation:

1. **Draw a stratified 200k working sample** from the full dataset
2. **Hold out 20%** (40k rows) as a final test set and keep it locked until the end
3. Use a **20k stratified tuning subset** for hyperparameter selection via stratified 3-fold CV
4. Report **mean ± SD** across folds for PR-AUC, F1, precision, and recall using the full 160k training set (CV experimental design)
5. Refit each selected model on the full 160k training set and evaluate once on the held-out 40k test set

In [8]:

# 7. MODEL TRAINING: CV-BASED COMPARISON + FINAL HELD-OUT TEST

from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split, cross_validate
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

df_model = df_clean.copy()
active_text_feature = TEXT_FEATURE

# --------------------------------------------------
# 0. Stratified 200k working sample
# --------------------------------------------------
WORKING_SAMPLE_SIZE = 200_000

df_work, _ = train_test_split(
    df_model,
    train_size=WORKING_SAMPLE_SIZE,
    stratify=df_model["SERIOUS"],
    random_state=42
)
df_model = df_work

print(f"Working sample: {len(df_model):,} rows  |  serious rate: {df_model['SERIOUS'].mean():.3f}")

# Derive active column lists from what actually exists in df_model
active_numeric_cols = [c for c in numeric_cols if c in df_model.columns]
active_categorical_cols = [c for c in categorical_cols if c in df_model.columns]
active_structured_cols = [c for c in structured_cols if c in df_model.columns]
active_binary_cols = [
    c for c in active_structured_cols
    if c not in (active_numeric_cols + active_categorical_cols)
]

# Build preprocessing pipeline
numeric_pipe_runtime = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe_runtime = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

structured_preprocess_runtime = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe_runtime, active_numeric_cols),
        ("cat", categorical_pipe_runtime, active_categorical_cols),
        ("binary", "passthrough", active_binary_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

text_preprocess_runtime = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=10,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True,
        norm="l2",
        strip_accents="unicode",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
    ))
])

full_preprocess_runtime = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess_runtime, active_structured_cols),
        ("text", text_preprocess_runtime, active_text_feature),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

print("Runtime feature setup:")
print("  Numeric:", len(active_numeric_cols))
print("  Categorical:", len(active_categorical_cols))
print("  Binary:", len(active_binary_cols))
print("  Structured total:", len(active_structured_cols))
print("  Text feature:", active_text_feature)

# --------------------------------------------------
# 1. Train / test split on working sample (80 / 20)
# --------------------------------------------------
df_train, df_test = train_test_split(
    df_model,
    test_size=0.20,
    stratify=df_model["SERIOUS"],
    random_state=42
)

y_train = df_train["SERIOUS"].astype(int)
y_test  = df_test["SERIOUS"].astype(int)

print(f"\nTrain set : {df_train.shape}  |  serious rate: {y_train.mean():.3f}")
print(f"Test  set : {df_test.shape}   |  serious rate: {y_test.mean():.3f}")

# --------------------------------------------------
# 1b. Tuning subset for GridSearchCV (~20k rows, ~12.5% of training set)
#     Keeps hyperparameter search fast; CV eval uses the full training set.
# --------------------------------------------------
TUNE_SIZE = 20_000

df_tune, _ = train_test_split(
    df_train,
    train_size=TUNE_SIZE,
    stratify=y_train,
    random_state=42
)
y_tune = df_tune["SERIOUS"].astype(int)

print(f"Tuning subset : {df_tune.shape}  |  serious rate: {y_tune.mean():.3f}")

# --------------------------------------------------
# 2. Models and parameter grids
# --------------------------------------------------
CV_SPLITS = 3

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

light_param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1.0]
    },
    "Decision Tree": {
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [5, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [100],
        "clf__max_depth": [10, 20]
    }
}

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=42)

cv_scoring = {
    "average_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

results = []
best_models = {}

print("Models to run:", list(models.keys()))

# --------------------------------------------------
# 3. Tune, CV evaluate, then confirm on held-out test
# --------------------------------------------------
for name, clf in models.items():
    print("\n" + "=" * 80)
    print("MODEL:", name)
    print("=" * 80)

    base_pipe = Pipeline([
        ("preprocess", full_preprocess_runtime),
        ("scale", MaxAbsScaler()),
        ("clf", clf)
    ])

    grid = GridSearchCV(
        estimator=base_pipe,
        param_grid=light_param_grids[name],
        scoring="average_precision",
        cv=cv,
        n_jobs=1,
        verbose=1,
        refit=True,
        error_score="raise"
    )

    # Stage A: Hyperparameter selection on tuning subset
    grid.fit(df_tune, y_tune)
    best_params = grid.best_params_
    print("Best params:", best_params)

    # Stage B: CV-based comparative evaluation with fixed params on full training set
    fixed_pipe = Pipeline([
        ("preprocess", full_preprocess_runtime),
        ("scale", MaxAbsScaler()),
        ("clf", clf)
    ])
    fixed_pipe.set_params(**best_params)

    cv_scores = cross_validate(
        estimator=fixed_pipe,
        X=df_train,
        y=y_train,
        scoring=cv_scoring,
        cv=cv,
        n_jobs=1,
        return_train_score=False,
        error_score="raise"
    )

    cv_pr_auc_mean = cv_scores["test_average_precision"].mean()
    cv_pr_auc_std = cv_scores["test_average_precision"].std()
    cv_f1_mean = cv_scores["test_f1"].mean()
    cv_f1_std = cv_scores["test_f1"].std()
    cv_precision_mean = cv_scores["test_precision"].mean()
    cv_precision_std = cv_scores["test_precision"].std()
    cv_recall_mean = cv_scores["test_recall"].mean()
    cv_recall_std = cv_scores["test_recall"].std()

    print(f"CV PR-AUC:    {cv_pr_auc_mean:.4f} ± {cv_pr_auc_std:.4f}")
    print(f"CV F1:        {cv_f1_mean:.4f} ± {cv_f1_std:.4f}")
    print(f"CV Precision: {cv_precision_mean:.4f} ± {cv_precision_std:.4f}")
    print(f"CV Recall:    {cv_recall_mean:.4f} ± {cv_recall_std:.4f}")

    # Stage C: Final refit on full training set and one-time held-out test
    fixed_pipe.fit(df_train, y_train)
    best_models[name] = fixed_pipe

    y_pred_test = fixed_pipe.predict(df_test)
    y_prob_test = fixed_pipe.predict_proba(df_test)[:, 1]

    test_pr_auc = average_precision_score(y_test, y_prob_test)
    test_f1 = f1_score(y_test, y_pred_test, zero_division=0)
    test_precision = precision_score(y_test, y_pred_test, zero_division=0)
    test_recall = recall_score(y_test, y_pred_test, zero_division=0)
    cm = confusion_matrix(y_test, y_pred_test)

    print(f"Test PR-AUC:    {test_pr_auc:.4f}")
    print(f"Test F1:        {test_f1:.4f}")
    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall:    {test_recall:.4f}")
    print("Test Confusion Matrix:")
    print(cm)

    results.append({
        "Model": name,
        "BestParams": best_params,
        "CV_PR_AUC_Mean": cv_pr_auc_mean,
        "CV_PR_AUC_SD": cv_pr_auc_std,
        "CV_F1_Mean": cv_f1_mean,
        "CV_F1_SD": cv_f1_std,
        "CV_Precision_Mean": cv_precision_mean,
        "CV_Precision_SD": cv_precision_std,
        "CV_Recall_Mean": cv_recall_mean,
        "CV_Recall_SD": cv_recall_std,
        "Test_PR_AUC": test_pr_auc,
        "Test_F1": test_f1,
        "Test_Precision": test_precision,
        "Test_Recall": test_recall,
        "ConfusionMatrix": cm
    })

summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "ConfusionMatrix"}
    for r in results
])

print("\nMODEL COMPARISON - CV experimental design (ranked by CV PR-AUC)")
print(summary_df.sort_values("CV_PR_AUC_Mean", ascending=False))

summary_df.to_csv(f"{OUTDIR}/supervised_model_comparison_raw_tfidf.csv", index=False)
print("\nStored best models:", list(best_models.keys()))


Working sample: 200,000 rows  |  serious rate: 0.199
Runtime feature setup:
  Numeric: 4
  Categorical: 4
  Binary: 66
  Structured total: 74
  Text feature: SYMPTOM_TEXT_CLEAN_NOLEAK

Train set : (160000, 115)  |  serious rate: 0.199
Test  set : (40000, 115)   |  serious rate: 0.199
Tuning subset : (20000, 115)  |  serious rate: 0.199
Models to run: ['Logistic Regression', 'Decision Tree', 'Random Forest']

MODEL: Logistic Regression
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best params: {'clf__C': 1.0}
CV PR-AUC:    0.8013 ± 0.0008
CV F1:        0.7025 ± 0.0012
CV Precision: 0.6090 ± 0.0029
CV Recall:    0.8299 ± 0.0021
Test PR-AUC:    0.8038
Test F1:        0.7054
Test Precision: 0.6105
Test Recall:    0.8353
Test Confusion Matrix:
[[27828  4232]
 [ 1308  6632]]

MODEL: Decision Tree
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best params: {'clf__max_depth': 10, 'clf__min_samples_split': 10}
CV PR-AUC:    0.6322 ± 0.0045
CV F1:        0.6130 ± 0.0009

## 8. Structured-Only vs Structured + Text

This tells us whether the text column meaningfully improves beyond what structured/clinical features alone capture.

In [9]:

# ============================================================
# 8. STRUCTURED-ONLY VS STRUCTURED + RAW TF-IDF
#    Both pipelines are fit on df_train and evaluated on df_test
#    Uses runtime preprocessors / column lists from Cell 7
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

comparison_results = []

# --------------------------------------------------
# Structured-only pipeline
# Rebuilt fresh from active_*_cols (Cell 7) to avoid stale-state issues
# --------------------------------------------------
structured_only_ct = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), active_numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), active_categorical_cols),
        ("binary", "passthrough", active_binary_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

structured_only_pipe = Pipeline([
    ("preprocess", structured_only_ct),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

# --------------------------------------------------
# Structured + text pipeline (uses full_preprocess_runtime from Cell 7)
# --------------------------------------------------
structured_text_pipe = Pipeline([
    ("preprocess", full_preprocess_runtime),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

for label, pipe in {
    "Structured only":         structured_only_pipe,
    "Structured + raw TF-IDF": structured_text_pipe
}.items():
    print("\nFitting:", label)
    pipe.fit(df_train, y_train)

    y_pred = pipe.predict(df_test)
    y_prob = pipe.predict_proba(df_test)[:, 1]

    comparison_results.append({
        "FeatureSet": label,
        "PR-AUC":     average_precision_score(y_test, y_prob),
        "F1":         f1_score(y_test, y_pred, zero_division=0),
        "Precision":  precision_score(y_test, y_pred, zero_division=0),
        "Recall":     recall_score(y_test, y_pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison_results)
print("\nSTRUCTURED vs TEXT COMPARISON — held-out test set")
print(comparison_df.sort_values("PR-AUC", ascending=False))

comparison_df.to_csv(f"{OUTDIR}/structured_vs_raw_tfidf_comparison.csv", index=False)



Fitting: Structured only

Fitting: Structured + raw TF-IDF

STRUCTURED vs TEXT COMPARISON — held-out test set
                FeatureSet    PR-AUC        F1  Precision    Recall
1  Structured + raw TF-IDF  0.803828  0.705382   0.610457  0.835264
0          Structured only  0.559951  0.533096   0.431081  0.698363


## 9. Feature Importance

Extract and interpret the features driving each model's predictions:
- **Logistic Regression**: signed coefficients (positive = predicts serious; negative = predicts non-serious)
- **Random Forest**: Gini-based feature importances (unsigned, relative)
- **Decision Tree**: Gini importances + which features appear in actual tree splits + exported rule text (depth ≤ 4)

Results are saved separately for structured and text features to allow easy comparison of clinical vs. symptom-text signal.

In [10]:

# 9A. FIT FINAL LOGISTIC REGRESSION MODEL FOR FEATURE IMPORTANCE
#      Trained on the full training set (df_train) so coefficients
#      reflect the model that was actually evaluated on the test set.

final_logreg_pipe = Pipeline([
    ("preprocess", full_preprocess_runtime),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

final_logreg_pipe.fit(df_train, y_train)

print("Final logistic regression model fitted on full training set.")


Final logistic regression model fitted on full training set.


In [11]:
# 9B. EXTRACT TOP FEATURES FROM LOGISTIC REGRESSION
#     (structured + raw TF-IDF)

import numpy as np
import pandas as pd

# --------------------------------------------------
# Get fitted preprocessors
# --------------------------------------------------
preprocessor = final_logreg_pipe.named_steps["preprocess"]
clf = final_logreg_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names = preprocessor.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names = preprocessor.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine all feature names
all_feature_names = np.concatenate([structured_feature_names, text_feature_names])

# Logistic regression coefficients
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "feature_type": ["structured"] * len(structured_feature_names) + ["text"] * len(text_feature_names)
})

# Top positive predictors of seriousness
top_positive = coef_df.sort_values("coefficient", ascending=False).head(50)

# Top negative predictors of seriousness
top_negative = coef_df.sort_values("coefficient", ascending=True).head(50)

# Top text-only predictors
top_text_positive = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=False).head(50)
top_text_negative = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=True).head(50)

print("\nTop positive predictors of SERIOUS=1")
print(top_positive[["feature", "coefficient", "feature_type"]])

print("\nTop negative predictors of SERIOUS=1")
print(top_negative[["feature", "coefficient", "feature_type"]])

print("\nTop POSITIVE text predictors")
print(top_text_positive[["feature", "coefficient"]])

print("\nTop NEGATIVE text predictors")
print(top_text_negative[["feature", "coefficient"]])

# Save results
coef_df.to_csv(f"{OUTDIR}/logreg_all_feature_coefficients_raw_tfidf.csv", index=False)
top_positive.to_csv(f"{OUTDIR}/logreg_top_positive_features_raw_tfidf.csv", index=False)
top_negative.to_csv(f"{OUTDIR}/logreg_top_negative_features_raw_tfidf.csv", index=False)
top_text_positive.to_csv(f"{OUTDIR}/logreg_top_positive_text_features_raw_tfidf.csv", index=False)
top_text_negative.to_csv(f"{OUTDIR}/logreg_top_negative_text_features_raw_tfidf.csv", index=False)



Top positive predictors of SERIOUS=1
               feature  coefficient feature_type
765               care     9.700947         text
865        clinic care     5.651175         text
2458                iv     5.598860         text
4443            stroke     5.294286         text
783              cause     5.157998         text
267          admission     4.796777         text
377       appendicitis     4.683751         text
1528       epinephrine     4.599528         text
5075              went     4.369262         text
4143              sent     4.263183         text
3966          required     4.231887         text
1526               epi     4.129231         text
768       care patient     3.981522         text
3564         presented     3.820546         text
761            cardiac     3.788667         text
4808      unresponsive     3.663628         text
1529            epipen     3.653312         text
3517         pneumonia     3.650780         text
475            arrived     3.63

In [12]:

# ============================================================
# 9C. RANDOM FOREST FEATURE IMPORTANCE
#      Fitted on the full training set (df_train)
# ============================================================

final_rf_pipe = Pipeline([
    ("preprocess", full_preprocess_runtime),
    ("scale", MaxAbsScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

final_rf_pipe.fit(df_train, y_train)

preprocessor_rf = final_rf_pipe.named_steps["preprocess"]
rf = final_rf_pipe.named_steps["clf"]

structured_feature_names_rf = preprocessor_rf.named_transformers_["structured"].get_feature_names_out()
text_feature_names_rf = preprocessor_rf.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()
all_feature_names_rf = np.concatenate([structured_feature_names_rf, text_feature_names_rf])

rf_importance_df = pd.DataFrame({
    "feature": all_feature_names_rf,
    "importance": rf.feature_importances_,
    "feature_type": ["structured"] * len(structured_feature_names_rf) + ["text"] * len(text_feature_names_rf)
}).sort_values("importance", ascending=False)

print("\nTop Random Forest features")
print(rf_importance_df.head(50))

rf_importance_df.to_csv(f"{OUTDIR}/random_forest_feature_importance_raw_tfidf.csv", index=False)



Top Random Forest features
                                  feature  importance feature_type
2                     num__ONSET_INTERVAL    0.050410   structured
102              binary__LAB_DATA_MISSING    0.042980   structured
1                            num__NUMDAYS    0.039826   structured
203                                 acute    0.015203         text
143   binary__LAB_DATA__vitals_procedures    0.014318   structured
144               binary__LAB_DATA__other    0.013300   structured
2060                             headache    0.013017         text
0                            num__AGE_YRS    0.011242   structured
4239                                 site    0.010958         text
2458                                   iv    0.010208         text
2373                            injection    0.010010         text
104       binary__HISTORY__cardiovascular    0.008309   structured
3564                            presented    0.008003         text
815                               

In [13]:

# 9D. FIT FINAL DECISION TREE MODEL
#      Fitted on the full training set (df_train)

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MaxAbsScaler

final_dt_pipe = Pipeline([
    ("preprocess", full_preprocess_runtime),
    ("scale", MaxAbsScaler()),
    ("clf", DecisionTreeClassifier(
        max_depth=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ))
])

final_dt_pipe.fit(df_train, y_train)

print("Final decision tree model fitted on full training set.")


Final decision tree model fitted on full training set.


In [14]:
# 9E. EXTRACT DECISION TREE FEATURE IMPORTANCE

# Get fitted objects
preprocessor_dt = final_dt_pipe.named_steps["preprocess"]
dt = final_dt_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names_dt = preprocessor_dt.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names_dt = preprocessor_dt.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine feature names
all_feature_names_dt = np.concatenate([structured_feature_names_dt, text_feature_names_dt])

# Importance values
dt_importances = dt.feature_importances_

# Put into dataframe
dt_importance_df = pd.DataFrame({
    "feature": all_feature_names_dt,
    "importance": dt_importances,
    "feature_type": ["structured"] * len(structured_feature_names_dt) + ["text"] * len(text_feature_names_dt)
})

# Sort descending
dt_importance_df = dt_importance_df.sort_values("importance", ascending=False)

# Top overall features
top_dt_features = dt_importance_df.head(50)

# Top text features only
top_dt_text_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "text"]
    .sort_values("importance", ascending=False)
    .head(50)
)

# Top structured features only
top_dt_structured_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "structured"]
    .sort_values("importance", ascending=False)
    .head(50)
)

print("\nTop overall Decision Tree features")
print(top_dt_features)

print("\nTop text Decision Tree features")
print(top_dt_text_features)

print("\nTop structured Decision Tree features")
print(top_dt_structured_features)

# Save results
dt_importance_df.to_csv(f"{OUTDIR}/decision_tree_feature_importance_raw_tfidf.csv", index=False)
top_dt_features.to_csv(f"{OUTDIR}/decision_tree_top_features_raw_tfidf.csv", index=False)
top_dt_text_features.to_csv(f"{OUTDIR}/decision_tree_top_text_features_raw_tfidf.csv", index=False)
top_dt_structured_features.to_csv(f"{OUTDIR}/decision_tree_top_structured_features_raw_tfidf.csv", index=False)



Top overall Decision Tree features
                                  feature  importance feature_type
2                     num__ONSET_INTERVAL    0.212383   structured
102              binary__LAB_DATA_MISSING    0.168649   structured
953                           concomitant    0.049833         text
144               binary__LAB_DATA__other    0.028527   structured
744                                called    0.028391         text
0                            num__AGE_YRS    0.019952   structured
765                                  care    0.019837         text
2458                                   iv    0.017819         text
1                            num__NUMDAYS    0.017000   structured
815                                 chest    0.013363         text
783                                 cause    0.012060         text
1555                           evaluation    0.010115         text
59                     cat__STATE_UNKNOWN    0.009575   structured
2924                      

In [15]:
# 9F. FEATURES USED IN DECISION TREE SPLITS

used_feature_idx = dt.tree_.feature
used_feature_idx = used_feature_idx[used_feature_idx >= 0]   # remove leaf markers (-2)

used_feature_names = all_feature_names_dt[used_feature_idx]
used_feature_names_unique = pd.Series(used_feature_names).value_counts().reset_index()
used_feature_names_unique.columns = ["feature", "num_splits"]

print("\nFeatures actually used in tree splits")
print(used_feature_names_unique.head(50))

used_feature_names_unique.to_csv(
    f"{OUTDIR}/decision_tree_features_used_in_splits_raw_tfidf.csv",
    index=False
)



Features actually used in tree splits
                       feature  num_splits
0                 num__AGE_YRS          66
1                 num__NUMDAYS          31
2                      patient          28
3          num__ONSET_INTERVAL          22
4                        covid          20
5                      vaccine          18
6                          arm          18
7                         care          18
8                         days          16
9                         went          15
10                    positive          15
11                   injection          14
12                        pain          13
13                      lasted          12
14                         did          12
15  binary__OTHER_MEDS_MISSING          12
16                       fever          12
17                        site          12
18                         day          12
19                        rash          12
20                      second          12
21             

In [16]:
# 9G. EXPORT DECISION TREE RULES

from sklearn.tree import export_text

tree_rules = export_text(
    dt,
    feature_names=list(all_feature_names_dt),
    max_depth=4
)

print(tree_rules)

with open(f"{OUTDIR}/decision_tree_rules_raw_tfidf.txt", "w") as f:
    f.write(tree_rules)


|--- num__ONSET_INTERVAL <= 0.00
|   |--- binary__LAB_DATA_MISSING <= 0.50
|   |   |--- binary__LAB_DATA__other <= 0.50
|   |   |   |--- mrna moderna <= 0.37
|   |   |   |   |--- available appropriate <= 0.39
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- available appropriate >  0.39
|   |   |   |   |   |--- truncated branch of depth 6
|   |   |   |--- mrna moderna >  0.37
|   |   |   |   |--- relationship <= 0.12
|   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |   |--- relationship >  0.12
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |--- binary__LAB_DATA__other >  0.50
|   |   |   |--- concomitant <= 0.21
|   |   |   |   |--- arm <= 0.05
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- arm >  0.05
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |--- concomitant >  0.21
|   |   |   |   |--- care <= 0.03
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- car